# 前向传播、反向传播和计算图核心知识点总结

## 一、核心概念定义

1. **计算图**

可视化变量与操作符依赖关系的工具：正方形代表变量，圆圈代表操作符，箭头表示数据流（前向传播时方向为“输入→输出”）。

2. **前向传播**

从输入层到输出层**顺序计算并存储各层中间变量**的过程，是获取模型预测结果与损失的基础。

3. **反向传播**

从输出层到输入层**利用链式法则计算参数梯度**的过程，是模型参数更新的核心依据。

## 二、前向传播：流程与公式（以带 $L_2$ 正则的单隐藏层MLP为例）

给定输入 $\mathbf{x} \in \mathbb{R}^d$ ，前向传播依次计算：

1. **隐藏层中间变量**： $\mathbf{z} = \mathbf{W}^{(1)} \mathbf{x}$ （ $\mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ 为隐藏层权重）；

2. **隐藏层激活值**： $\mathbf{h} = \phi(\mathbf{z})$ （ $\phi$ 为激活函数）；

3. **输出层变量**： $\mathbf{o} = \mathbf{W}^{(2)} \mathbf{h}$ （ $\mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$ 为输出层权重）；

4. **损失项**： $L = l(\mathbf{o}, y)$ （ $l$ 为损失函数， $y$ 为样本标签）；

5.  $L_2$  **正则项**： $s = \frac{\lambda}{2} \left(|\mathbf{W}^{(1)}|_F^2 + |\mathbf{W}^{(2)}|_F^2\right)$ （ $\lambda$ 为正则化超参数， $|\cdot|_F$ 为Frobenius范数）；

6. **目标函数**： $J = L + s$ （模型最终优化的函数）。

## 三、反向传播：原理与步骤（基于链式法则）

从目标函数 $J$ 出发，**反向遍历计算图**，依次计算参数梯度：

1. 目标函数对损失/正则项的梯度： $\frac{\partial J}{\partial L} = 1$ ， $\frac{\partial J}{\partial s} = 1$ ；

2. 对输出层变量的梯度： $\frac{\partial J}{\partial \mathbf{o}} = \frac{\partial L}{\partial \mathbf{o}} \in \mathbb{R}^q$ ；

3. 正则项对参数的梯度： $\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)}$ ， $\frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}$ ；

4. 对输出层权重 $\mathbf{W}^{(2)}$ 的梯度： $\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$ ；

5. 对隐藏层激活值 $\mathbf{h}$ 的梯度： $\frac{\partial J}{\partial \mathbf{h}} = {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}} \in \mathbb{R}^h$ ；

6. 对隐藏层中间变量 $\mathbf{z}$ 的梯度： $\frac{\partial J}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \odot \phi'(\mathbf{z})$ （ $\odot$ 为按元素乘法， $\phi'$ 为激活函数导数）；

7. 对输入层权重 $\mathbf{W}^{(1)}$ 的梯度： $\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ 。

## 四、训练中的关键关系与内存问题

1. **前向与反向的依赖**

    - 前向传播提供反向传播所需的中间变量（如 $\mathbf{h}$ 、 $\mathbf{z}$ ）；

    - 反向传播提供前向传播更新参数所需的梯度。

2. **内存占用差异**

    - 训练：需保留前向传播的中间值，内存占用与**网络层数、批量大小**正相关（大批量+深层网络易出现内存不足）；

    - 预测：仅需前向传播，无需保留中间值，内存占用更低。

## 五、核心结论

- 前向传播是“从输入到输出的计算流”，反向传播是“从输出到输入的梯度流”，二者相互依赖；

- 反向传播的本质是链式法则在计算图上的应用；

- 训练的内存消耗高于预测，源于中间变量的存储需求。